<div style="background:linear-gradient(135deg,#4c0519 0%,#be123c 55%,#fb7185 100%);border-radius:18px;padding:32px 30px;color:#fff;font-family:Inter,Segoe UI,sans-serif">
  <div style="font-size:12px;letter-spacing:3px;color:#fecdd3;font-weight:700;text-transform:uppercase">Chapter 157 &middot; Communicating Results &middot; Report 1 of 3</div>
  <div style="font-size:32px;font-weight:900;line-height:1.1;margin:10px 0 6px">Executive Report: Quarterly Sales Performance</div>
  <div style="font-size:15px;color:#ffe4e6;max-width:760px;line-height:1.6">The executive one-pager. Lead with the answer, back it with three or four numbers and one chart, and end with a recommendation. This notebook turns the sales data into a finished, downloadable Word report.</div>
</div>

In [ ]:
import numpy as np, pandas as pd
import matplotlib as mpl, matplotlib.pyplot as plt
# A clean house style for report-ready figures: no chartjunk, strong titles, muted grid.
mpl.rcParams.update({"figure.dpi":110,"font.size":11,"axes.spines.top":False,"axes.spines.right":False,
    "axes.grid":True,"grid.alpha":0.22,"axes.titleweight":"bold","axes.titlesize":12.5,
    "axes.titlelocation":"left","axes.titlepad":10})
ROSE, INK, MUT, GR, RD = "#be123c", "#1a2138", "#64748b", "#16a34a", "#dc2626"
BASE = "https://raw.githubusercontent.com/johnfisher-ai/Statistics-Data-Science-AI-Visual-Book/main/data/"
fn = "communicating-insights--company-data.xlsx"
def load(sheet):
    try: return pd.read_excel("../../data/" + fn, sheet_name=sheet)
    except FileNotFoundError: return pd.read_excel(BASE + fn, sheet_name=sheet)
from docx import Document
from docx.shared import Inches, Pt, RGBColor
from docx.enum.text import WD_ALIGN_PARAGRAPH
from pathlib import Path
import tempfile
ROSE_DOC, GREY_DOC = RGBColor(0xBE,0x12,0x3C), RGBColor(0x64,0x74,0x8B)
def new_report(title, subtitle):
    doc = Document()
    for s in doc.sections:                       # US Letter, sensible margins
        s.page_width, s.page_height = Inches(8.5), Inches(11)
        s.left_margin = s.right_margin = Inches(1); s.top_margin = s.bottom_margin = Inches(0.9)
    t = doc.add_heading(title, level=0)
    for r in t.runs: r.font.color.rgb = ROSE_DOC
    sp = doc.add_paragraph(subtitle); sp.runs[0].italic = True; sp.runs[0].font.color.rgb = GREY_DOC
    return doc
def h2(doc, text):
    hd = doc.add_heading(text, level=1)
    for r in hd.runs: r.font.color.rgb = ROSE_DOC
def bullets(doc, items):
    for it in items: doc.add_paragraph(it, style="List Bullet")
def df_table(doc, df):                           # df cells should already be display strings
    tb = doc.add_table(rows=1, cols=len(df.columns)); tb.style = "Light Grid Accent 1"
    for j, c in enumerate(df.columns):
        cell = tb.rows[0].cells[j]; cell.text = str(c)
        for r in cell.paragraphs[0].runs: r.bold = True
    for _, row in df.iterrows():
        cells = tb.add_row().cells
        for j, c in enumerate(df.columns): cells[j].text = str(row[c])
    return tb
def add_fig(doc, fig, width=6.3):
    p = Path(tempfile.mkdtemp()) / "fig.png"; fig.savefig(p, dpi=150, bbox_inches="tight"); plt.close(fig)
    doc.add_picture(str(p), width=Inches(width)); doc.paragraphs[-1].alignment = WD_ALIGN_PARAGRAPH.CENTER
def save_report(doc, name):                       # write into the book repo when present, else the cwd (Colab)
    outdir = Path("../../reports") if Path("../../reports").exists() else Path(".")
    outdir.mkdir(exist_ok=True); path = outdir / name; doc.save(path)
    print("wrote", path.resolve()); return path

## Step 1 &middot; Load and analyze
Read the monthly sales sheet and compute the handful of numbers an executive actually needs: the headline total, the trend, and the regional split.

In [ ]:
sales = load("SalesMonthly")
sales["quarter"] = "Q" + pd.PeriodIndex(sales.month, freq="M").quarter.astype(str)
total_rev = sales.revenue.sum(); total_orders = int(sales.orders.sum()); aov = total_rev/total_orders
byq = sales.groupby("quarter").revenue.sum()
growth = byq["Q4"]/byq["Q1"] - 1
byregion = sales.groupby("region").revenue.sum().sort_values(ascending=False)
print(f"total revenue ${total_rev:,.0f} | orders {total_orders:,} | AOV ${aov:.2f} | Q4 vs Q1 {growth:+.1%}")
print(byregion.round(0).to_string())

## Step 2 &middot; Build the two visuals
One trend chart and one ranked bar. Each title states the point so the figures stand alone.

In [ ]:
fig_trend, ax = plt.subplots(figsize=(7.6, 3.2))
m = sales.groupby("month").revenue.sum()/1000
ax.plot(m.index, m.values, color=ROSE, lw=2.4, marker="o", ms=3)
ax.set_title(f"Revenue grew {growth:+.0%} across 2024"); ax.set_ylabel("monthly revenue ($k)")
ax.tick_params(axis="x", rotation=45, labelsize=7); plt.tight_layout(); plt.show()

fig_reg, ax = plt.subplots(figsize=(7.6, 3.0))
r = byregion.sort_values()/1000
ax.barh(r.index, r.values, color=ROSE)
for y,v in enumerate(r.values): ax.text(v+4, y, f"${v:,.0f}k", va="center", fontsize=9)
ax.set_title("North leads; Central trails by a third"); ax.set_xlim(0, r.max()*1.16)
ax.grid(axis="y", visible=False); plt.tight_layout(); plt.show()

## Step 3 &middot; Assemble the Word report
Now compose the document: a bottom-line-up-front summary, a KPI table, the two figures, the findings, and a clear recommendation. Everything is generated from the numbers above, so it can never drift from the analysis.

In [ ]:
doc = new_report("Quarterly Sales Performance, 2024", "Prepared for the leadership team  |  Confidential")

h2(doc, "Bottom line")
doc.add_paragraph(f"Revenue reached ${total_rev/1e6:.2f}M in 2024, up {growth:.0%} from the first quarter to the "
                  f"fourth. Growth was broad but uneven: North is now a third larger than Central. We recommend "
                  f"shifting a share of Q1 marketing spend toward Central and South to close the regional gap.")

h2(doc, "The numbers at a glance")
kpi = pd.DataFrame({"Metric":["Total revenue","Orders","Average order value","Q4 vs Q1 growth","Top region"],
                    "Value":[f"${total_rev:,.0f}", f"{total_orders:,}", f"${aov:.2f}", f"{growth:+.0%}", "North"]})
df_table(doc, kpi)

h2(doc, "Revenue trend and regional split")
add_fig(doc, fig_trend); add_fig(doc, fig_reg)

h2(doc, "What we found")
bullets(doc, [f"Revenue rose every quarter, from ${byq['Q1']/1000:,.0f}k in Q1 to ${byq['Q4']/1000:,.0f}k in Q4.",
              f"North (${byregion.iloc[0]/1000:,.0f}k) and West (${byregion.iloc[1]/1000:,.0f}k) drove over 45% of revenue.",
              f"Central (${byregion.iloc[-1]/1000:,.0f}k) remains the clear laggard and the biggest upside."])

h2(doc, "Recommendation")
doc.add_paragraph("Rebalance a portion of marketing investment toward the underperforming Central and South regions "
                  "in Q1 2025, and set a target of narrowing the North-to-Central revenue gap from 39% to under 25%.")

path = save_report(doc, "report-1-quarterly-performance.docx")

### Wrap-up
That is the executive genre: the recommendation appears in the first sentence, supported by a five-row KPI table, two self-titled figures, and three findings, then a concrete next step. The whole document is built from the analysis variables, so re-running it on next quarter's data regenerates the report automatically.